# PCA Surrogate — Hyperparameter Optimization

Optuna TPE (with median pruning) over four model variants:

| Variant         | Class         | Geometry input | Loss                                 |
|-----------------|---------------|---------------:|--------------------------------------|
| `pca_4D`        | `PcaMLP`      |             4D | coeff-MSE                            |
| `pca_peak_4D`   | `PcaPeakMLP`  |             4D | coeff-MSE + `peak_loss_weight` * peak-MSE |
| `pca_8D`        | `PcaMLP`      |             8D | coeff-MSE                            |
| `pca_peak_8D`   | `PcaPeakMLP`  |             8D | coeff-MSE + `peak_loss_weight` * peak-MSE |

**Per-trial scoring (option a — ensemble target).** For each trial:
1. Train all `K_FOLDS` fold models with the proposed config.
2. For each fold's val set, predict PCA coefficients with **every** fold model and average them — this is the ensemble prediction (matches the inference-time recipe used in the presentation).
3. Reconstruct spectra from the averaged coefficients, compute val recon MSE per fold, then mean across folds.
4. Trial score = `mean(ensemble_recon_rmse) + STD_PENALTY * std(...)`.

Note: the ensemble metric uses every fold model on every fold's val set, including models that saw the sample during training. This slightly inflates the absolute number vs. true held-out performance, but the *ranking* across configs is what HPO needs and matches the deployment-time inference flow exactly.

**Ranges in the K sweep.** For each variant, the optimization is run independently for every K in `K_SWEEP`. Optuna does not pick K — you do, by reading the per-K table at the bottom.

---

## Hyperparameter glossary (and starting ranges)

| HP                  | Type            | Default range                  | Notes                                                                  |
|---------------------|-----------------|--------------------------------|------------------------------------------------------------------------|
| `hidden_dim`        | categorical     | `[128, 192, 256, 384, 512]`    | trunk width. ~1k samples — bigger isn't obviously better.              |
| `n_layers`          | categorical     | `[2, 3, 4]`                    | trunk depth. >4 risks vanishing gradients without residuals.           |
| `p`                 | float           | `[0.0, 0.3]`                   | dropout. Higher when you suspect overfitting (val>>train).             |
| `lr`                | log-uniform     | `[1e-4, 1e-2]`                 | AdamW learning rate.                                                   |
| `wd`                | log-uniform     | `[1e-5, 1e-2]`                 | AdamW weight decay.                                                    |
| `batch_size`        | categorical     | `[16, 32, 64]`                 | smaller = more SGD noise = mild regularization on small data.          |
| `peak_loss_weight`* | log-uniform     | `[1e-3, 1.0]`                  | *peak variants only*. Balances PCA-MSE (raw scale) vs peak-MSE (standardized scale). Optuna will find what minimizes recon. |

Discrete params are categorical so the enqueued seed config lands cleanly on grid points.

## 1. Top-level configuration
Everything you'd normally tweak lives in this one cell.

In [3]:
#### WHAT TO RUN ####
# Choose any subset of: 'pca_4D', 'pca_peak_4D', 'pca_8D', 'pca_peak_8D'
MODEL_VARIANTS = ['pca_4D', 'pca_peak_4D', 'pca_8D', 'pca_peak_8D']
K_SWEEP        = [16, 18, 20, 22, 25]

NAME           = 'optimization_v1'   # subfolder name for results

#### REPRODUCIBILITY ####
SEED           = 1234                # used for fold split, dataloader shuffle, Optuna sampler

#### CV + TRAINING ####
TEST_RATIO     = 0.15                # held out, untouched in HPO
K_FOLDS        = 4
EPOCHS         = 500
PATIENCE       = 100

#### OPTUNA ####
TRIALS         = 40                  # per (variant, K) — total = TRIALS * len(K_SWEEP) * len(MODEL_VARIANTS)
STD_PENALTY    = 0.3                 # score = mean_rmse + STD_PENALTY * std_rmse

#### SEARCH RANGES ####
# Discrete -> categorical lists; continuous -> (low, high). lr/wd/peak_loss_weight are sampled log-uniform.
SEARCH_RANGES = {
    'hidden_dim':       [128, 192, 256, 384, 512],
    'n_layers':         [2, 3, 4],
    'p':                (0.0, 0.3),
    'lr':               (1e-4, 1e-2),
    'wd':               (1e-5, 1e-2),
    'batch_size':       [16, 32, 64],
    'peak_loss_weight': (1e-3, 1.0),  # only used by peak variants
}

#### SEED CONFIGS (enqueued as trial 0 per variant so TPE starts from a sensible baseline) ####
BEST_KNOWN = {
    'pca_4D':      {'hidden_dim': 256, 'n_layers': 3, 'p': 0.036, 'lr': 7.5e-3, 'wd': 4.3e-4, 'batch_size': 16},
    'pca_8D':      {'hidden_dim': 256, 'n_layers': 3, 'p': 0.07,  'lr': 5e-3,   'wd': 8e-4,   'batch_size': 16},
    'pca_peak_4D': {'hidden_dim': 256, 'n_layers': 3, 'p': 0.05,  'lr': 5e-3,   'wd': 5e-4,   'batch_size': 16, 'peak_loss_weight': 0.1},
    'pca_peak_8D': {'hidden_dim': 256, 'n_layers': 3, 'p': 0.05,  'lr': 5e-3,   'wd': 5e-4,   'batch_size': 16, 'peak_loss_weight': 0.1},
}

## 2. Imports and data load
The 4D and 8D PCA features are imported as separate symbols, and the multi-peak features are loaded from the pickle.

In [2]:
import os, sys, json, math, copy, time, pickle, tempfile
import numpy as np
import torch
import optuna

# Make the kernel see ML_project/ as cwd + import root, regardless of where the
# notebook lives. pca_4D.py / pca_8D.py use './data/batch2' relative to cwd, and
# multi_peak_ft.pkl is also written there, so all relative paths resolve once
# we chdir up to ML_project/.
_ML_ROOT = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..'))
if _ML_ROOT not in sys.path:
    sys.path.insert(0, _ML_ROOT)
os.chdir(_ML_ROOT)
print(f"cwd -> {os.getcwd()}")

from util.classes.PCADataset import PCADataset
from util.classes.PCAPeakDataset import PCAPeakDataset
from util.classes.PcaMLP import PcaMLP, train_pca_regression as train_pca_only, create_dataloader_kfold as kfold_loaders_pca
from util.classes.PcaPeakMLP import PcaPeakMLP, train_pca_regression as train_pca_peak, create_dataloader_kfold as kfold_loaders_pca_peak
from util.model_optimization import generate_kfold

from preprocessing.pca_4D import pca_ft_4D, data_config_4D
from preprocessing.pca_8D import pca_ft_8D, data_config_8D

with open('multi_peak_ft.pkl', 'rb') as f:
    multi_peak_ft = pickle.load(f)

print(f"4D PCA: {pca_ft_4D['absorption_features'].shape[0]} samples x {pca_ft_4D['absorption_features'].shape[1]} PCs (geom dim {pca_ft_4D['geometry_table'].shape[1]})")
print(f"8D PCA: {pca_ft_8D['absorption_features'].shape[0]} samples x {pca_ft_8D['absorption_features'].shape[1]} PCs (geom dim {pca_ft_8D['geometry_table'].shape[1]})")
print(f"Multi-peak amp table: {multi_peak_ft['absorption_features'].shape}  (peak_count={multi_peak_ft['peak_count']})")

max_K = max(K_SWEEP)
for name, src in [('pca_4D', pca_ft_4D), ('pca_8D', pca_ft_8D)]:
    have = src['absorption_features'].shape[1]
    assert have >= max_K, f"{name} only has {have} PCs but K_SWEEP requires {max_K}. Re-run preprocessing with K>={max_K}."

VARIANT_INFO = {
    'pca_4D':      {'feat': pca_ft_4D, 'has_peak': False, 'input_dim': pca_ft_4D['geometry_table'].shape[1]},
    'pca_peak_4D': {'feat': pca_ft_4D, 'has_peak': True,  'input_dim': pca_ft_4D['geometry_table'].shape[1]},
    'pca_8D':      {'feat': pca_ft_8D, 'has_peak': False, 'input_dim': pca_ft_8D['geometry_table'].shape[1]},
    'pca_peak_8D': {'feat': pca_ft_8D, 'has_peak': True,  'input_dim': pca_ft_8D['geometry_table'].shape[1]},
}
for v in MODEL_VARIANTS:
    assert v in VARIANT_INFO, f"Unknown variant {v!r}. Choose from {list(VARIANT_INFO)}."

RESULTS_ROOT = os.path.join('optimization_results', NAME)
os.makedirs(RESULTS_ROOT, exist_ok=True)
print(f"\nResults root: {RESULTS_ROOT}")

c:\Users\robert\.conda\envs\torch_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cwd -> c:\Users\robert\Code\capstone\capstone_SiC_gratings\ML_project
4D PCA: 1947 samples x 26 PCs (geom dim 4)
8D PCA: 1947 samples x 26 PCs (geom dim 8)
Multi-peak amp table: (1947, 12)  (peak_count=4)

Results root: optimization_results\optimization_v1


## 3. Helpers — dataset construction, suggestion logic, ensemble metric

In [4]:
def _json_default(o):
    if isinstance(o, np.ndarray): return o.tolist()
    if isinstance(o, (np.integer, np.floating)): return o.item()
    raise TypeError(f"Not JSON serializable: {type(o).__name__}")


def fmt_sci(x: float) -> str:
    m, e = f"{x:.2e}".split('e')
    return f"{m.rstrip('0').rstrip('.')}e{int(e)}"


def config_name(cfg: dict) -> str:
    base = (f"dim{cfg['hidden_dim']}_layers{cfg['n_layers']}"
            f"_p{int(round(cfg['p']*100)):03d}"
            f"_lr{fmt_sci(cfg['lr'])}_wd{fmt_sci(cfg['wd'])}_bs{cfg['batch_size']}")
    if 'peak_loss_weight' in cfg:
        base += f"_pw{fmt_sci(cfg['peak_loss_weight'])}"
    return base


def build_dataset(variant: str, K: int):
    """Construct the right dataset for a variant, sliced to K PCs."""
    info = VARIANT_INFO[variant]
    feat = info['feat']
    geom_df = feat['geometry_table']
    pca_coeffs = feat['absorption_features'].iloc[:, :K]
    pca_arts = feat['absorption_pca']

    if not info['has_peak']:
        return PCADataset(
            geom_df, pca_coeffs, pca_arts,
            normalize_geom=True, normalize_feat=False,
        )
    amp_df = multi_peak_ft['absorption_features'].loc[geom_df.index]
    return PCAPeakDataset(
        geom_df, pca_coeffs, amp_df, pca_arts,
        normalize_geom=True, normalize_pca=False, normalize_amp=True,
    )


def suggest_config(trial: optuna.trial.Trial, has_peak: bool) -> dict:
    cfg = {
        'hidden_dim': trial.suggest_categorical('hidden_dim', SEARCH_RANGES['hidden_dim']),
        'n_layers':   trial.suggest_categorical('n_layers',   SEARCH_RANGES['n_layers']),
        'p':          trial.suggest_float('p',  *SEARCH_RANGES['p']),
        'lr':         trial.suggest_float('lr', *SEARCH_RANGES['lr'], log=True),
        'wd':         trial.suggest_float('wd', *SEARCH_RANGES['wd'], log=True),
        'batch_size': trial.suggest_categorical('batch_size', SEARCH_RANGES['batch_size']),
    }
    if has_peak:
        cfg['peak_loss_weight'] = trial.suggest_float(
            'peak_loss_weight', *SEARCH_RANGES['peak_loss_weight'], log=True
        )
    return cfg


def seed_everything(seed: int):
    """Pin python/numpy/torch RNGs so a given seed reproduces a given run."""
    import random
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

In [5]:
def evaluate_config(
        cfg: dict,
        variant: str,
        K: int,
        kf_indices: list,
        trial: optuna.trial.Trial | None = None,
):
    """Train K_FOLDS fold models, then evaluate the *ensemble* (mean of all fold
    models' coefficient predictions) on each fold's val set, and return the
    mean+std of the per-fold ensemble recon RMSE as the trial score.

    Pruning hook: after each fold finishes, the running mean of per-fold *single-
    model* val_recon_rmse (from the early-stopping checkpoint) is reported so the
    MedianPruner can kill obviously bad trials before they finish all K folds.
    """
    info = VARIANT_INFO[variant]
    has_peak = info['has_peak']
    input_dim = info['input_dim']

    data = build_dataset(variant, K)
    wl_tensor = torch.tensor(np.asarray(data.wl), dtype=torch.float32)

    fold_models = []
    per_fold_single = {  # tracked from each fold's best epoch (for pruning + diagnostics)
        'best_val_coeff_mse': [],
        'best_val_recon_mse': [],
        'best_val_recon_rmse': [],
        'best_epoch': [],
        'epochs_trained': [],
    }

    for fold_i, (train_idx, val_idx) in enumerate(kf_indices):
        seed_everything(SEED + fold_i)
        if has_peak:
            model = PcaPeakMLP(K, cfg['hidden_dim'], cfg['n_layers'], cfg['p'],
                               n_peaks=multi_peak_ft['peak_count'], input_dim=input_dim)
            train_loader, val_loader = kfold_loaders_pca_peak(
                data, SEED, train_idx, val_idx, cfg['batch_size'])
        else:
            model = PcaMLP(K, cfg['hidden_dim'], cfg['n_layers'], cfg['p'], input_dim=input_dim)
            train_loader, val_loader = kfold_loaders_pca(
                data, SEED, train_idx, val_idx, cfg['batch_size'])

        with tempfile.TemporaryDirectory() as tmpdir:
            if has_peak:
                _, history = train_pca_peak(
                    model, data, train_loader, val_loader,
                    EPOCHS, cfg['lr'], cfg['wd'], PATIENCE, tmpdir,
                    peak_loss_weight=cfg['peak_loss_weight'],
                )
            else:
                _, history = train_pca_only(
                    model, data, train_loader, val_loader,
                    EPOCHS, cfg['lr'], cfg['wd'], PATIENCE, tmpdir,
                )

        best_ep = history['stop_epoch']
        coeff_mse = history.get('val_loss_pca', history.get('val_loss'))[best_ep]
        recon_mse = history['val_loss_recon'][best_ep]

        per_fold_single['best_val_coeff_mse'].append(float(coeff_mse))
        per_fold_single['best_val_recon_mse'].append(float(recon_mse))
        per_fold_single['best_val_recon_rmse'].append(float(math.sqrt(recon_mse)))
        per_fold_single['best_epoch'].append(int(best_ep))
        per_fold_single['epochs_trained'].append(int(len(history['val_loss_recon'])))

        model.eval()
        fold_models.append(model)

        if trial is not None:
            trial.report(float(np.mean(per_fold_single['best_val_recon_rmse'])), step=fold_i)
            if trial.should_prune():
                raise optuna.TrialPruned()

    # ---- Ensemble metric: average all fold models' predictions on each fold's val set ----
    ensemble_per_fold = {'recon_mse': [], 'recon_rmse': [], 'peak_mae': [], 'peak_loc_um': []}
    with torch.no_grad():
        for fold_i, (_train_idx, val_idx) in enumerate(kf_indices):
            geom_val = data.geom[val_idx]
            true_coeffs = data.pca[val_idx] if has_peak else data.feat[val_idx]

            preds = []
            for m in fold_models:
                out = m(geom_val)
                preds.append(out[0] if has_peak else out)
            ensemble_coeffs = torch.stack(preds, dim=0).mean(dim=0)

            pred_spec = data.reconstruct_spectrum(ensemble_coeffs)
            true_spec = data.reconstruct_spectrum(true_coeffs)
            recon_mse = ((pred_spec - true_spec) ** 2).mean().item()

            pred_peak = pred_spec.max(dim=-1)
            true_peak = true_spec.max(dim=-1)
            peak_mae = (pred_peak.values - true_peak.values).abs().mean().item()
            peak_loc = (wl_tensor[pred_peak.indices] - wl_tensor[true_peak.indices]).abs().mean().item()

            ensemble_per_fold['recon_mse'].append(float(recon_mse))
            ensemble_per_fold['recon_rmse'].append(float(math.sqrt(recon_mse)))
            ensemble_per_fold['peak_mae'].append(float(peak_mae))
            ensemble_per_fold['peak_loc_um'].append(float(peak_loc))

    summary = {
        'ensemble': {k: {'mean': float(np.mean(v)), 'std': float(np.std(v))} for k, v in ensemble_per_fold.items()},
        'single_fold': {k: {'mean': float(np.mean(v)), 'std': float(np.std(v))} for k, v in per_fold_single.items()},
    }
    score = summary['ensemble']['recon_rmse']['mean'] + STD_PENALTY * summary['ensemble']['recon_rmse']['std']
    return summary, float(score)

## 4. Main optimization loop
One Optuna study per `(variant, K)`. Results saved incrementally to `optimization_results/<NAME>/<variant>/K{K}.json` so you can interrupt without losing progress.

In [6]:
all_results = {}   # all_results[variant][K] = winner record + trial log
start_wall = time.time()

for variant in MODEL_VARIANTS:
    info = VARIANT_INFO[variant]
    has_peak = info['has_peak']
    variant_dir = os.path.join(RESULTS_ROOT, variant)
    os.makedirs(variant_dir, exist_ok=True)
    all_results[variant] = {}

    # Same K-fold split for every K within this variant. The split depends only
    # on dataset length, so as long as the variant's geometry table is fixed it
    # is stable across K's (slicing PCs doesn't change the sample count).
    base_data = build_dataset(variant, max_K)
    kf_indices, test_indices = generate_kfold(base_data, K_FOLDS, SEED, TEST_RATIO)

    print('\n' + '#'*72)
    print(f"# VARIANT: {variant}  (input_dim={info['input_dim']}, has_peak={has_peak})")
    print(f"# {len(base_data) - len(test_indices)} train+val samples / {len(test_indices)} held-out test (untouched)")
    print('#'*72)

    for K in K_SWEEP:
        print('\n' + '='*72)
        print(f" {variant} | K = {K} | {TRIALS} trials")
        print('='*72)

        trial_cache: dict = {}

        def objective(trial: optuna.trial.Trial) -> float:
            cfg = suggest_config(trial, has_peak)
            name = config_name(cfg)
            if name in trial_cache:
                return trial_cache[name]['score']
            try:
                summary, score = evaluate_config(cfg, variant, K, kf_indices, trial=trial)
            except optuna.TrialPruned:
                raise
            except Exception as e:
                print(f"    trial failed: {e}")
                return float('inf')
            trial_cache[name] = {'name': name, 'config': cfg, 'score': score, 'summary': summary}
            ens = summary['ensemble']
            print(f"    {name}\n      score={score:.6g}  ens_rmse={ens['recon_rmse']['mean']:.6g}"
                  f"  peak_loc_mae={ens['peak_loc_um']['mean']:.6g} um")
            return score

        sampler = optuna.samplers.TPESampler(seed=SEED)
        pruner  = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1)
        study   = optuna.create_study(direction='minimize', sampler=sampler, pruner=pruner)

        if variant in BEST_KNOWN:
            seed_cfg = dict(BEST_KNOWN[variant])
            if has_peak and 'peak_loss_weight' not in seed_cfg:
                seed_cfg['peak_loss_weight'] = 0.1
            study.enqueue_trial(seed_cfg)

        study.optimize(objective, n_trials=TRIALS)

        completed = [r for r in trial_cache.values()]
        if not completed:
            print(f"  !! No completed trials for {variant} K={K}; skipping.")
            continue
        best = min(completed, key=lambda r: r['score'])
        n_pruned   = sum(1 for t in study.trials if t.state == optuna.trial.TrialState.PRUNED)
        n_complete = sum(1 for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE)

        record = {
            'variant': variant,
            'K': K,
            'best_config': best['config'],
            'best_name': best['name'],
            'best_score': best['score'],
            'best_summary': best['summary'],
            'meta': {
                'seed': SEED, 'test_ratio': TEST_RATIO, 'k_folds': K_FOLDS,
                'epochs': EPOCHS, 'patience': PATIENCE, 'trials': TRIALS,
                'trials_completed': n_complete, 'trials_pruned': n_pruned,
                'std_penalty': STD_PENALTY, 'input_dim': info['input_dim'],
                'search_ranges': SEARCH_RANGES, 'has_peak': has_peak,
                'best_known_enqueued': BEST_KNOWN.get(variant),
                'metric': 'ensemble_recon_rmse_mean + std_penalty * std',
            },
            'trial_log': [{'name': r['name'], 'config': r['config'], 'score': r['score']} for r in completed],
        }
        out_path = os.path.join(variant_dir, f'K{K}.json')
        with open(out_path, 'w') as f:
            json.dump(record, f, indent=4, default=_json_default)
        all_results[variant][K] = record
        print(f"  -> best score {best['score']:.6g}  ({n_complete} complete / {n_pruned} pruned)")
        print(f"     saved {out_path}")

elapsed_min = (time.time() - start_wall) / 60.0
print(f"\nDONE. Total wall time: {elapsed_min:.1f} min")

[I 2026-04-30 21:22:46,459] A new study created in memory with name: no-name-a7d32504-e89d-4855-b7a7-bcb37cbfb795



########################################################################
# VARIANT: pca_4D  (input_dim=4, has_peak=False)
# 1655 train+val samples / 292 held-out test (untouched)
########################################################################

 pca_4D | K = 16 | 40 trials
Early stopping triggered at epoch 249
Early stopping triggered at epoch 231
Early stopping triggered at epoch 316


[I 2026-04-30 21:25:28,453] Trial 0 finished with value: 0.019492560250815243 and parameters: {'hidden_dim': 256, 'n_layers': 3, 'p': 0.036, 'lr': 0.0075, 'wd': 0.00043, 'batch_size': 16}. Best is trial 0 with value: 0.019492560250815243.


Early stopping triggered at epoch 267
    dim256_layers3_p004_lr7.5e-3_wd4.3e-4_bs16
      score=0.0194926  ens_rmse=0.0191241  peak_loc_mae=0.00414236 um
Early stopping triggered at epoch 304
Early stopping triggered at epoch 272
Early stopping triggered at epoch 342


[I 2026-04-30 21:27:36,017] Trial 1 finished with value: 0.02998362700730663 and parameters: {'hidden_dim': 384, 'n_layers': 4, 'p': 0.28744180610511155, 'lr': 0.005647617424572882, 'wd': 0.00011842729505689425, 'batch_size': 64}. Best is trial 0 with value: 0.019492560250815243.


...

Early stopping triggered at epoch 225
    dim384_layers3_p003_lr4.24e-3_wd2.73e-3_bs16_pw1.39e-2
      score=0.0176633  ens_rmse=0.0173882  peak_loc_mae=0.00408614 um
Early stopping triggered at epoch 251
Early stopping triggered at epoch 239
Early stopping triggered at epoch 229


[I 2026-05-01 09:04:19,035] Trial 34 finished with value: 0.01689398197749129 and parameters: {'hidden_dim': 384, 'n_layers': 3, 'p': 0.025507171042407694, 'lr': 0.004384773021883884, 'wd': 0.0005667614478191849, 'batch_size': 16, 'peak_loss_weight': 0.031477547403493664}. Best is trial 30 with value: 0.01686063250418584.


Early stopping triggered at epoch 216
    dim384_layers3_p003_lr4.38e-3_wd5.67e-4_bs16_pw3.15e-2
      score=0.016894  ens_rmse=0.0166003  peak_loc_mae=0.00355556 um
Early stopping triggered at epoch 271
Early stopping triggered at epoch 294


[W 2026-05-01 09:07:18,832] Trial 35 failed with parameters: {'hidden_dim': 384, 'n_layers': 3, 'p': 0.1209497951402626, 'lr': 0.004669390671070339, 'wd': 0.0008276559788250377, 'batch_size': 16, 'peak_loss_weight': 0.0409298693319372} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "c:\Users\robert\.conda\envs\torch_env\Lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "C:\Users\robert\AppData\Local\Temp\ipykernel_24224\2497082209.py", line 35, in objective
    summary, score = evaluate_config(cfg, variant, K, kf_indices, trial=trial)
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\robert\AppData\Local\Temp\ipykernel_24224\3686779218.py", line 46, in evaluate_config
    _, history = train_pca_peak(
                 ^^^^^^^^^^^^^^^
  File "c:\Users\robert\Code\capstone\capstone_SiC_gratings\ML_proje

KeyboardInterrupt: 

## 5. Per-variant summary table
For each variant, the best ensemble RMSE / peak-loc MAE per K — pick the K with the best balance for your reporting.

In [ ]:
import pandas as pd

rows = []
for variant, by_k in all_results.items():
    for K, rec in by_k.items():
        ens = rec['best_summary']['ensemble']
        cfg = rec['best_config']
        rows.append({
            'variant': variant,
            'K': K,
            'score': rec['best_score'],
            'ens_recon_rmse': ens['recon_rmse']['mean'],
            'ens_recon_rmse_std': ens['recon_rmse']['std'],
            'ens_peak_mae': ens['peak_mae']['mean'],
            'ens_peak_loc_um': ens['peak_loc_um']['mean'],
            'hidden_dim': cfg['hidden_dim'],
            'n_layers': cfg['n_layers'],
            'p': round(cfg['p'], 4),
            'lr': cfg['lr'],
            'wd': cfg['wd'],
            'batch_size': cfg['batch_size'],
            'peak_loss_weight': cfg.get('peak_loss_weight'),
        })

summary_df = pd.DataFrame(rows).sort_values(['variant', 'K']).reset_index(drop=True)
with pd.option_context('display.max_columns', None, 'display.width', 200, 'display.float_format', '{:.6g}'.format):
    print(summary_df.to_string(index=False))

summary_path = os.path.join(RESULTS_ROOT, 'summary_table.csv')
summary_df.to_csv(summary_path, index=False)
print(f"\nSaved {summary_path}")